# Text-to-SQL: Seq2Seq vs. Causal LM

The same task, the same data, the same metric, two architectures:
**BART-base** (encoder-decoder) and **GPT-2** (decoder-only), both fine-tuned on
`gretelai/synthetic_text_to_sql` and scored by exact match on 2,000 held-out
questions.

| Model | Raw match | Normalized match |
|---|---|---|
| **BART-base** (seq2seq) | 21.85% | **22.30%** |
| GPT-2 (causal) | 18.90% | 20.45% |

BART wins by ~2 points, which is the expected direction — translating a question
into SQL is a sequence-to-sequence problem, and an encoder that sees the whole
question bidirectionally before decoding suits it better than a left-to-right LM.

**The more interesting result is that 22% badly understates both models.** The
error analysis below shows most "wrong" predictions are SQL that would return the
right answer; exact match penalises them for cosmetic differences. That is the
real finding here — the metric, not the models, is the weak link.

## 1. The dataset

`gretelai/synthetic_text_to_sql` — natural-language questions paired with gold
SQL, split into train / dev / test.

In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

ds = load_dataset("gretelai/synthetic_text_to_sql", split='train')
df = pd.DataFrame(ds)

df = df.rename(columns={'sql_prompt': 'question', 'sql_context': 'schema', 'sql': 'query'})
df = df[['question', 'schema', 'query']]

total_samples = 22500
if len(df) > total_samples:
    df = df.sample(n=total_samples, random_state=42).reset_index(drop=True)

train_dev_df, test_df = train_test_split(df, test_size=2000, random_state=42)
train_df, dev_df = train_test_split(train_dev_df, test_size=500, random_state=42)

print(f"Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")

sample = train_df.iloc[0]
print("-" * 50)
print("Sample Data Entry:")
print(f"Question: {sample['question']}")
print(f"Schema:   {sample['schema']}")
print(f"SQL:      {sample['query']}")
print("-" * 50)

## 2. Fine-tuning BART-base

`facebook/bart-base` as a conditional generation model: the question is the
encoder input, the SQL is the decoder target.

`sqlparse` is imported for the normalization step rather than for training — the
raw and normalized metrics below differ only in whether SQL is canonicalised
before comparison.

In [ ]:
import torch
import tqdm
import sqlparse
from torch.utils.data import Dataset, DataLoader
from transformers import BartTokenizer, BartForConditionalGeneration, AdamW

class SQLDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        input_text = f"Question: {row['question']} Context: {row['schema']}"
        
        inputs = self.tokenizer(
            input_text, 
            max_length=self.max_len, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        
        with self.tokenizer.as_target_tokenizer():
            targets = self.tokenizer(
                row['query'], 
                max_length=self.max_len, 
                padding="max_length", 
                truncation=True, 
                return_tensors="pt"
            )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels,
            "original_sql": row['query']
        }

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

EPOCHS = 3           
BATCH_SIZE = 8      
LEARNING_RATE = 2e-5 


train_dataset = SQLDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {EPOCHS} epochs...")


model.train()
for epoch in range(EPOCHS):
  
    loop = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

  
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
    
        optimizer.zero_grad()
        outputs.loss.backward()
        optimizer.step()
        
       
        loop.set_postfix(loss=outputs.loss.item())

print("Training Finished.")

def normalize_sql(sql):
    if not isinstance(sql, str): return ""
    sql = sql.strip().rstrip(';')
    try:
        sql = sqlparse.format(sql, keyword_case='lower', identifier_case='lower')
    except:
        pass
    return " ".join(sql.replace('\n', ' ').split())

def evaluate(model, dataframe, tokenizer, num_samples=100):
    model.eval()
    eval_loader = DataLoader(SQLDataset(dataframe, tokenizer), batch_size=4)
    raw_matches, norm_matches, total = 0, 0, 0
    
    with torch.no_grad():
        for batch in eval_loader:
            if total >= num_samples: break
            
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            generated_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128)
            decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            
            for pred, gold in zip(decoded_preds, batch["original_sql"]):
                if pred.strip() == gold.strip(): raw_matches += 1
                if normalize_sql(pred) == normalize_sql(gold): norm_matches += 1
                total += 1

    print(f"Raw Accuracy: {raw_matches / total:.2%}")
    print(f"Normalized Accuracy: {norm_matches / total:.2%}")

evaluate(model, dev_df, tokenizer)

### The evaluation loop

Greedy decoding, then string comparison against the gold query.

In [ ]:
def evaluate(model, dataframe, tokenizer, num_samples=500):
    model.eval()
    eval_loader = DataLoader(SQLDataset(dataframe, tokenizer), batch_size=4)
    raw_matches, norm_matches, total = 0, 0, 0
    
    with torch.no_grad():
        for batch in eval_loader:
            if total >= num_samples: break
            
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            generated_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128)
            decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            
            for pred, gold in zip(decoded_preds, batch["original_sql"]):
                if pred.strip() == gold.strip(): raw_matches += 1
                if normalize_sql(pred) == normalize_sql(gold): norm_matches += 1
                total += 1

    print(f"Raw Accuracy: {raw_matches / total:.2%}")
    print(f"Normalized Accuracy: {norm_matches / total:.2%}")

evaluate(model, dev_df, tokenizer)

### Inspecting individual predictions

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class SQLDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        input_text = f"Question: {row['question']} Context: {row['schema']}"
        
        inputs = self.tokenizer(
            input_text, 
            max_length=self.max_len, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        
        with self.tokenizer.as_target_tokenizer():
            targets = self.tokenizer(
                row['query'], 
                max_length=self.max_len, 
                padding="max_length", 
                truncation=True, 
                return_tensors="pt"
            )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels,
            "original_sql": row['query'],
            "question": row['question']  
        }


print("Dataset class updated. Running analysis...")

model.eval()
subset = dev_df.sample(n=5)
loader = DataLoader(SQLDataset(subset, tokenizer), batch_size=1)

print("\n 5 Random Examples ")

with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        generated_ids = model.generate(input_ids=input_ids, attention_mask=mask, max_length=100)
        pred = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        gold = batch["original_sql"][0]
        question = batch["question"][0]

        norm_pred = normalize_sql(pred)
        norm_gold = normalize_sql(gold)

        print(f"Question:   {question}")
        print(f"Prediction: {pred}")
        print(f"Gold:       {gold}")

        if norm_pred == norm_gold:
            print("Result: CORRECT")
        else:
            print("Result: WRONG")
            if len(pred) < 5:
                print("Error Type: Output too short.")
            elif "join" in gold.lower() and "join" not in pred.lower():
                print("Error Type: Missed JOIN.")
            elif "where" in gold.lower() and "where" not in pred.lower():
                print("Error Type: Missed WHERE clause.")
            else:
                print("Error Type: Logic or Column mismatch.")
        
        print("x" * 100)

### BART on the full test set

**2,000 samples: 21.85% raw, 22.30% normalized.**

Normalization buys only half a point, which says the failures are not mostly
whitespace and casing — they are structural.

In [ ]:
print(f"Evaluating {len(test_df)} samples from Test set...")

model.eval()
eval_loader = DataLoader(SQLDataset(test_df, tokenizer), batch_size=16)

raw_correct = 0
norm_correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm.tqdm(eval_loader):
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        
        generated_ids = model.generate(input_ids=input_ids, attention_mask=mask, max_length=128)
        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        golds = batch["original_sql"]
        
        for pred, gold in zip(preds, golds):
            if pred.strip() == gold.strip():
                raw_correct += 1
            
            if normalize_sql(pred) == normalize_sql(gold):
                norm_correct += 1
            
            total += 1

print(f"Raw Match Accuracy: {raw_correct/total:.2%}")
print(f"Normalized Match Accuracy: {norm_correct/total:.2%}")

## 3. Fine-tuning GPT-2 on the same data

A decoder-only model has no separate encoder, so the question and the SQL are
concatenated into one sequence and the model learns to continue it. Three epochs
at 5e-5, batch size 8.

In [ ]:
import torch
import sys
import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForSeq2Seq
)
from datasets import Dataset

# Assuming train_df, dev_df, test_df are already loaded as pandas DataFrames


train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Right padding for training

def format_gpt2_input(example):
  
    prefix = f"Question: {example['question']} Context: {example['schema']} SQL: "
    full_text = prefix + example['query'] + tokenizer.eos_token
    
    return {
        "text": full_text, 
        "prompt": prefix, 
        "gold_sql": example['query']
    }

# Apply formatting
lm_train = train_dataset.map(format_gpt2_input)
lm_dev = dev_dataset.map(format_gpt2_input)
lm_test = test_dataset.map(format_gpt2_input)

def tokenize_and_mask_labels(examples):

    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
        return_attention_mask=True
    )
    
    input_ids_list = tokenized["input_ids"]
    labels_list = []
    
    for i, input_ids in enumerate(input_ids_list):
        prompt = examples["prompt"][i]
        
        
        prompt_ids = tokenizer(prompt, truncation=True, max_length=512)["input_ids"]
        prompt_len = len(prompt_ids)
        
     
        label = list(input_ids)
        mask_len = min(prompt_len, len(label))
        
        # Mask the prefix/prompt part
        label[:mask_len] = [-100] * mask_len
        labels_list.append(label)
    
    tokenized["labels"] = labels_list
    return tokenized

# Keep only necessary columns for the Trainer
cols_to_keep = ["input_ids", "attention_mask", "labels"]
tokenized_train = lm_train.map(tokenize_and_mask_labels, batched=True, remove_columns=[c for c in lm_train.column_names if c not in cols_to_keep])
tokenized_dev = lm_dev.map(tokenize_and_mask_labels, batched=True, remove_columns=[c for c in lm_dev.column_names if c not in cols_to_keep])


model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.use_cache = False  # Disable caching for training to save memory

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir="./gpt2_text2sql_results",
    num_train_epochs=3,              
    per_device_train_batch_size=8,   
    learning_rate=5e-5,              
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    evaluation_strategy="no",        
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator,
)

print("Starting GPT-2 Training...")
train_result = trainer.train()

print("="*40)
print(f"Training Finished.")
print(f"Total Time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Final Training Loss: {train_result.metrics['train_loss']:.4f}")
print("="*40)


model.save_pretrained("./final_gpt2_text2sql")
tokenizer.save_pretrained("./final_gpt2_text2sql")
print("Model saved to ./final_gpt2_text2sql")

## 4. GPT-2's numbers, and what the errors really are

**DEV: 19.40% raw / 21.40% normalized. TEST: 18.90% raw / 20.45% normalized.**

The sampled predictions are where the metric falls apart. Of five DEV examples
marked `WRONG`:

| Predicted | Gold | Actually wrong? |
|---|---|---|
| `SELECT region, AVG(word_count) FROM Articles GROUP BY region;` | `... AVG(word_count) as avg_word_count ...` | **No** — same result set, different column alias |
| `... construction_date > '2010-12-31'` | `... construction_date > '2010-01-01'` | Yes, but only a date boundary — and "after 2010" is genuinely ambiguous |
| `MIN(clearing_height_feet)` | `MIN(clearance_height_feet)` | Yes — a one-character schema hallucination |
| `SELECT AVG(posts.comment_time), AVG(posts.comment_time) ...` | a correlated subquery over four joins | Yes — real failure on a genuinely hard query |

So the failure modes are three different things that exact match reports
identically: **cosmetic** (aliases, whitespace), **near-miss** (a wrong column
name or a date boundary), and **structural** (the model cannot build a multi-join
aggregate).

Only the third is a modelling problem. Execution accuracy — running both queries
against the schema and comparing result sets — would separate them, and both
models would score considerably higher. Exact match is cheap to compute and easy
to misread.

In [ ]:

def normalize_sql(sql):
    if not isinstance(sql, str): return ""
    sql = sql.replace(";", "").strip().lower()
    return " ".join(sql.split())

def evaluate_gpt2(dataset, dataset_name):
    print(f"\n--- Evaluating on {dataset_name} Set ({len(dataset)} samples) ---")
    tokenizer.padding_side = "left" # Switch to left for generation
    model.eval()
    
    predictions, references, questions = [], [], []
    
    correct_raw = 0
    correct_norm = 0
    total = 0

    for i in tqdm.tqdm(range(len(dataset))):
        item = dataset[i]
        prompt = item["prompt"]
        gold = item["gold_sql"]
        question = item["question"]
        
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=128, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if prompt in generated_text:
            generated_sql = generated_text.split(prompt)[1].strip()
        else:
            generated_sql = generated_text 
            
        generated_sql = generated_sql.split('\n')[0].strip()

        predictions.append(generated_sql)
        references.append(gold)
        questions.append(question)
        
        if generated_sql.strip() == gold.strip():
            correct_raw += 1
        if normalize_sql(generated_sql) == normalize_sql(gold):
            correct_norm += 1
        total += 1
        
    print(f"{dataset_name} Raw Accuracy: {correct_raw/total:.2%}")
    print(f"{dataset_name} Normalized Accuracy: {correct_norm/total:.2%}")
    
    return predictions, references, questions

preds_dev, golds_dev, questions_dev = evaluate_gpt2(lm_dev, "DEV")
preds_test, golds_test, questions_test = evaluate_gpt2(lm_test, "TEST")


print("\n--- 5 Random Examples from DEV ---")
import random

indices = random.sample(range(len(preds_dev)), 5)

for i in indices:
    pred = preds_dev[i]
    gold = golds_dev[i]
    q = questions_dev[i]
    
    norm_pred = normalize_sql(pred)
    norm_gold = normalize_sql(gold)
    
    print(f"Question:   {q}")
    print(f"Prediction: {pred}")
    print(f"Gold:       {gold}")
    
    if norm_pred == norm_gold:
        print("Result: CORRECT")
    else:
        print("Result: WRONG")
        if len(pred) < 5: print("Error: Output too short.")
        elif "join" in gold.lower() and "join" not in pred.lower(): print("Error: Missed JOIN.")
        elif "where" in gold.lower() and "where" not in pred.lower(): print("Error: Missed WHERE.")
        else: print("Error: Logic/Syntax mismatch.")
    print("-" * 50)

### Sampled predictions

Two of five correct, and both correct ones are byte-identical to the gold query.
The pattern holds: simple aggregates land, anything needing a `WHERE` derived
from an implicit constraint (*"fans who attend sports events"* → a subquery over
`Events`) does not.

In [ ]:
print("\n 5 Random Examples from DEV ")
import random

indices = random.sample(range(len(preds_dev)), 5)

for i in indices:
    pred = preds_dev[i]
    gold = golds_dev[i]
    q = questions_dev[i]
    
    norm_pred = normalize_sql(pred)
    norm_gold = normalize_sql(gold)
    
    print(f"Question:   {q}")
    print(f"Prediction: {pred}")
    print(f"Gold:       {gold}")
    
    if norm_pred == norm_gold:
        print("Result: CORRECT")
    else:
        print("Result: WRONG")
        if len(pred) < 5: print("Error: Output too short.")
        elif "join" in gold.lower() and "join" not in pred.lower(): print("Error: Missed JOIN.")
        elif "where" in gold.lower() and "where" not in pred.lower(): print("Error: Missed WHERE.")
        else: print("Error: Logic/Syntax mismatch.")
    print("x" * 100)